# CLIP4CAD Inference Demo

Text-to-CAD retrieval with 3D visualization.

**Features:**
- Search 166K CAD models with natural language
- Compare BRep vs Point Cloud embeddings
- 3D STEP file visualization inline
- Switch between checkpoints for comparison

## Usage
1. Run cells 1-5 to setup (first run computes embeddings ~10 min)
2. Use `text_to_cad("your query")` to search

---
## Cell 1: Imports

In [1]:
import sys
from pathlib import Path

# Add project root
sys.path.insert(0, 'D:/Defect_Det/MMCAD/MMCAD')

import json
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from omegaconf import OmegaConf

# Visualization
import plotly.graph_objects as go
from IPython.display import display, HTML

# PythonOCC for STEP loading
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.IFSelect import IFSelect_RetDone
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopAbs import TopAbs_FACE
from OCC.Core.BRep import BRep_Tool
from OCC.Core.TopLoc import TopLoc_Location

# Model
from clip4cad.ablations.models import CLIP4CAD_GFA_Ablation
from clip4cad.ablations.configs import get_ablation_config
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


---
## Cell 2: Configuration

In [2]:
# =============================================================================
# PATHS
# =============================================================================
PATHS = {
    "step_dir": "D:/Defect_Det/MMCAD/data/extracted_step_files",
    "metadata_csv": "D:/Defect_Det/MMCAD/data/169k.csv",
    "splits_dir": "D:/Defect_Det/MMCAD/data/aligned/splits",
    "uid_mapping": "D:/Defect_Det/MMCAD/data/aligned/uid_mapping.json",
    "brep_file": "c:/Users/User/Desktop/brep_features.h5",
    "pc_file": "c:/Users/User/Desktop/pc_embeddings_full.h5",
    "text_file": "c:/Users/User/Desktop/text_embeddings.h5",  # REQUIRED for text-grounded gallery encoding
    "config_path": "D:/Defect_Det/MMCAD/MMCAD/configs/model/clip4cad_gfa.yaml",
    "embeddings_cache": "D:/Defect_Det/MMCAD/data/aligned/global_embeddings_cache.h5",
}

# =============================================================================
# MODEL CHECKPOINT - Change this to compare different checkpoints
# =============================================================================
CHECKPOINT_PATH = "D:/Defect_Det/MMCAD/MMCAD/notebooks/outputs/ablations/asymmetric_grounding_5-1/checkpoint_epoch35.pt"
ABLATION_TYPE = "asymmetric_grounding"

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Ablation: {ABLATION_TYPE}")

Checkpoint: D:/Defect_Det/MMCAD/MMCAD/notebooks/outputs/ablations/asymmetric_grounding_5-1/checkpoint_epoch35.pt
Ablation: asymmetric_grounding


---
## Cell 3: Load Model from Checkpoint

In [3]:
def load_model(checkpoint_path: str, ablation_type: str):
    """Load model from checkpoint."""
    print(f"Loading model from: {checkpoint_path}")
    
    # Load config
    config = get_ablation_config(PATHS["config_path"], ablation_type)
    
    # Create model
    model = CLIP4CAD_GFA_Ablation(config)
    
    # Load weights
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    
    model = model.to(device).eval()
    
    print(f"  Epoch: {ckpt.get('epoch', '?')}")
    print(f"  Stage: {ckpt.get('stage', '?')}")
    print(f"  Parameters: {model.count_parameters():,}")
    
    return model, config

# Load model
model, config = load_model(CHECKPOINT_PATH, ABLATION_TYPE)

Loading model from: D:/Defect_Det/MMCAD/MMCAD/notebooks/outputs/ablations/asymmetric_grounding_5-1/checkpoint_epoch35.pt
  [Ablation] Using LEARNED confidence
  Epoch: 35
  Stage: 2
  Parameters: 3,267,971


---
## Cell 4: Load Metadata & UIDs

In [4]:
# Load all split UIDs
print("Loading UIDs from all splits...")
all_uids = []
for split in ["train", "val", "test"]:
    split_file = Path(PATHS["splits_dir"]) / f"{split}_uids.txt"
    if split_file.exists():
        uids = np.loadtxt(split_file, dtype=int).tolist()
        all_uids.extend(uids)
        print(f"  {split}: {len(uids):,} UIDs")

all_uids = sorted(set(all_uids))  # Remove duplicates, sort
print(f"Total unique UIDs: {len(all_uids):,}")

# Load metadata
print("\nLoading metadata...")
metadata_df = pd.read_csv(PATHS["metadata_csv"])
metadata_df = metadata_df.set_index("uid")
print(f"  Metadata rows: {len(metadata_df):,}")

# Load UID mapping
print("\nLoading UID mapping...")
with open(PATHS["uid_mapping"], "r") as f:
    uid_mapping = json.load(f)
print(f"  Mapping entries: {len(uid_mapping):,}")

# Create UID to index mapping for fast lookup
uid_to_idx = {uid: i for i, uid in enumerate(all_uids)}
print(f"\nReady! {len(all_uids):,} models available for search.")

Loading UIDs from all splits...
  train: 133,105 UIDs
  val: 16,638 UIDs
  test: 16,639 UIDs
Total unique UIDs: 166,382

Loading metadata...


C:\Users\User\AppData\Local\Temp\ipykernel_12612\3518527565.py:16: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_df = pd.read_csv(PATHS["metadata_csv"])


  Metadata rows: 169,460

Loading UID mapping...
  Mapping entries: 166,382

Ready! 166,382 models available for search.


---
## Cell 5: Compute/Load Global Embeddings

In [5]:
def compute_embeddings(model, all_uids, uid_mapping, batch_size=64):
    """
    Compute z_brep and z_pc embeddings for all UIDs using FULL FORWARD PASS.
    
    CRITICAL: Uses text-grounded embeddings (same as training/evaluation).
    
    The model was trained to align:
    - z_text (from text -> feature slots -> confidence-weighted aggregation)
    - z_brep (from TEXT-grounded importance -> weighted aggregation)
    - z_pc (from TEXT-grounded importance -> weighted aggregation)
    
    Using encode_geometry() with self-grounding produces DIFFERENT embeddings
    than training, which causes retrieval to fail. We must use the full forward
    pass with actual text data for correct gallery encoding.
    """
    print("Computing global embeddings using FULL FORWARD PASS (text-grounded)...")
    print("This takes ~10 minutes on first run, then cached.")
    
    # Open ALL HDF5 files including TEXT (required for text-grounded embeddings)
    brep_h5 = h5py.File(PATHS["brep_file"], "r")
    pc_h5 = h5py.File(PATHS["pc_file"], "r")
    text_h5 = h5py.File(PATHS["text_file"], "r")  # CRITICAL: text features for grounding
    
    all_brep_emb = []
    all_pc_emb = []
    valid_uids = []
    
    model.eval()
    
    with torch.no_grad():
        for i in tqdm(range(0, len(all_uids), batch_size), desc="Computing embeddings"):
            batch_uids = all_uids[i:i+batch_size]
            
            # Build batch with ALL features including text
            batch_brep_face = []
            batch_brep_edge = []
            batch_brep_face_mask = []
            batch_brep_edge_mask = []
            batch_pc = []
            batch_text_emb = []
            batch_text_mask = []
            batch_valid = []
            
            for uid in batch_uids:
                uid_str = str(uid)
                if uid_str not in uid_mapping:
                    continue
                    
                mapping = uid_mapping[uid_str]
                brep_idx = mapping["brep_idx"]
                pc_idx = mapping["pc_idx"]
                text_idx = mapping.get("text_idx", brep_idx)  # Fallback to brep_idx if text_idx not present
                
                try:
                    # Load ALL features including text
                    face_feat = brep_h5["face_features"][brep_idx]
                    edge_feat = brep_h5["edge_features"][brep_idx]
                    face_mask = brep_h5["face_masks"][brep_idx]
                    edge_mask = brep_h5["edge_masks"][brep_idx]
                    pc_feat = pc_h5["local_features"][pc_idx]
                    
                    # CRITICAL: Load text embeddings for text-grounded forward pass
                    text_emb = text_h5["desc_embeddings"][text_idx]
                    text_mask = text_h5["desc_masks"][text_idx]
                    
                    batch_brep_face.append(face_feat)
                    batch_brep_edge.append(edge_feat)
                    batch_brep_face_mask.append(face_mask)
                    batch_brep_edge_mask.append(edge_mask)
                    batch_pc.append(pc_feat)
                    batch_text_emb.append(text_emb)
                    batch_text_mask.append(text_mask)
                    batch_valid.append(uid)
                except Exception as e:
                    continue
            
            if not batch_valid:
                continue
            
            # Convert to tensors
            batch = {
                "brep_face_features": torch.tensor(np.stack(batch_brep_face), dtype=torch.float32, device=device),
                "brep_edge_features": torch.tensor(np.stack(batch_brep_edge), dtype=torch.float32, device=device),
                "brep_face_mask": torch.tensor(np.stack(batch_brep_face_mask), dtype=torch.bool, device=device),
                "brep_edge_mask": torch.tensor(np.stack(batch_brep_edge_mask), dtype=torch.bool, device=device),
                "pc_features": torch.tensor(np.stack(batch_pc), dtype=torch.float32, device=device),
                "desc_embedding": torch.tensor(np.stack(batch_text_emb), dtype=torch.float32, device=device),
                "desc_mask": torch.tensor(np.stack(batch_text_mask), dtype=torch.float32, device=device),
                "use_cached_embeddings": True,
                "use_cached_brep_features": True,
                "use_cached_pc_features": True,
            }
            
            # FULL FORWARD PASS (same as training/evaluation)
            outputs = model(batch)
            
            # Extract TEXT-GROUNDED embeddings (NOT self-grounded)
            z_brep = F.normalize(outputs["z_brep"].float(), dim=-1)
            z_pc = F.normalize(outputs["z_pc"].float(), dim=-1)
            
            all_brep_emb.append(z_brep.cpu())
            all_pc_emb.append(z_pc.cpu())
            valid_uids.extend(batch_valid)
    
    brep_h5.close()
    pc_h5.close()
    text_h5.close()
    
    all_brep_emb = torch.cat(all_brep_emb, dim=0)
    all_pc_emb = torch.cat(all_pc_emb, dim=0)
    
    print(f"Computed TEXT-GROUNDED embeddings for {len(valid_uids):,} samples")
    print(f"  BRep: {all_brep_emb.shape}")
    print(f"  PC: {all_pc_emb.shape}")
    
    return valid_uids, all_brep_emb, all_pc_emb


def save_embeddings(uids, brep_emb, pc_emb, cache_path):
    """Save embeddings to cache."""
    print(f"Saving embeddings to {cache_path}...")
    with h5py.File(cache_path, "w") as f:
        f.create_dataset("uids", data=np.array(uids, dtype=np.int64))
        f.create_dataset("brep_embeddings", data=brep_emb.numpy())
        f.create_dataset("pc_embeddings", data=pc_emb.numpy())
    print("Saved!")


def load_embeddings(cache_path):
    """Load embeddings from cache."""
    print(f"Loading cached embeddings from {cache_path}...")
    with h5py.File(cache_path, "r") as f:
        uids = f["uids"][:].tolist()
        brep_emb = torch.tensor(f["brep_embeddings"][:])
        pc_emb = torch.tensor(f["pc_embeddings"][:])
    print(f"Loaded {len(uids):,} embeddings")
    return uids, brep_emb, pc_emb


# Check for cache or compute
cache_path = Path(PATHS["embeddings_cache"])

# NOTE: Delete cache file to recompute with text-grounded embeddings!
# If you previously computed with self-grounding, the cache is incorrect.
if cache_path.exists():
    print("WARNING: Using cached embeddings. Delete cache to recompute with text-grounding:")
    print(f"  {cache_path}")
    gallery_uids, brep_embeddings, pc_embeddings = load_embeddings(cache_path)
else:
    gallery_uids, brep_embeddings, pc_embeddings = compute_embeddings(model, all_uids, uid_mapping)
    save_embeddings(gallery_uids, brep_embeddings, pc_embeddings, cache_path)

# Create lookup
gallery_uid_to_idx = {uid: i for i, uid in enumerate(gallery_uids)}
print(f"\nGallery ready: {len(gallery_uids):,} models")

Computing global embeddings using FULL FORWARD PASS (text-grounded)...
This takes ~10 minutes on first run, then cached.


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'c:/Users/User/Desktop/text_embeddings.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

---
## Cell 6: Text Query Encoding

In [ ]:
# Initialize text encoder for live queries (if not already loaded)
if model.tokenizer is None:
    print("Initializing text encoder for live queries...")
    from transformers import AutoModel, AutoTokenizer
    
    model_name = config.encoders.text.get("model_name", "microsoft/Phi-4-mini-instruct")
    
    # Load tokenizer
    model.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if model.tokenizer.pad_token is None:
        model.tokenizer.pad_token = model.tokenizer.eos_token
        model.tokenizer.pad_token_id = model.tokenizer.eos_token_id
    
    # Load LLM with optimized attention
    for attn_impl in ["flash_attention_2", "sdpa", None]:
        try:
            kwargs = {"torch_dtype": torch.float16, "trust_remote_code": True}
            if attn_impl:
                kwargs["attn_implementation"] = attn_impl
            model.text_llm = AutoModel.from_pretrained(model_name, **kwargs)
            print(f"  Using {attn_impl or 'default'} attention")
            break
        except Exception as e:
            if attn_impl:
                print(f"  {attn_impl} not available")
            continue
    
    model.text_llm = model.text_llm.to(device).eval()
    for param in model.text_llm.parameters():
        param.requires_grad = False
    print("  Text encoder ready!")


def encode_text_query(model, query: str, return_intermediate: bool = False):
    """
    Encode a text query to embedding space using live LLM encoding.
    Matches the precompute_text_embeddings_csv.py pipeline exactly.
    """
    model.eval()
    tokenizer = model.tokenizer
    
    # Tokenize (same as precompute script)
    encoded = tokenizer(
        query,
        padding="max_length",
        truncation=True,
        max_length=256,  # Match training max_desc_len
        return_tensors="pt",
    )
    
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    
    with torch.no_grad():
        # Get LLM embeddings (same as precompute: model() -> last_hidden_state)
        text_output = model.text_llm(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        # Convert to float32 (same as precompute: .float())
        text_features = text_output.last_hidden_state.float()
        
        # Project to unified space
        X_text = model.projection.project_text(text_features)
        
        # Parse into feature slots
        # Training uses desc_mask which is attention_mask saved as float (1.0=valid, 0.0=pad)
        # CrossAttentionLayer.forward does: attn_mask = ~key_padding_mask.bool()
        # So pass attention_mask directly (1=valid → True, 0=pad → False after .bool())
        text_mask = attention_mask  # Keep as int/float, matches training
        T_feat, confidence, attn_weights = model.parse_text_features(X_text, text_mask)
        
        # Global text embedding (confidence-weighted) - same as model.forward
        conf_sum = confidence.sum(dim=-1, keepdim=True).clamp(min=1e-8)
        conf_norm = confidence / conf_sum
        z_text = torch.sum(conf_norm.unsqueeze(-1) * T_feat, dim=1)
        
        # Project and normalize
        z_text_proj = model.global_proj_head(z_text)
        z_text_proj = F.normalize(z_text_proj.float(), dim=-1)
    
    if return_intermediate:
        return {
            "z_text": z_text_proj.cpu(),
            "T_feat": T_feat.cpu(),
            "confidence": confidence.cpu(),
            "X_text": X_text.cpu(),
            "text_features": text_features.cpu(),
        }
    return z_text_proj.cpu()


# =============================================================================
# DIAGNOSTIC: Compare live encoding vs pre-computed (verify pipeline match)
# =============================================================================
print("=" * 70)
print("DIAGNOSTIC: Comparing live encoding vs pre-computed embeddings")
print("=" * 70)

# Find a valid sample UID that exists in metadata
sample_uid = None
for uid in gallery_uids[:100]:
    if uid in metadata_df.index:
        sample_uid = uid
        break

if sample_uid is None:
    print("Warning: No matching UID found in metadata, using test query only")
    sample_text = "A mechanical part with cylindrical features"
else:
    metadata_row = metadata_df.loc[sample_uid]
    sample_title = str(metadata_row.get("title", "") or "")
    sample_desc = str(metadata_row.get("description", "") or "")
    
    # Format text same as precompute script
    if sample_title:
        sample_text = f"{sample_title}. {sample_desc}"
    else:
        sample_text = sample_desc if sample_desc else "A CAD model"
    
    print(f"\nSample UID: {sample_uid}")
    print(f"Text (first 100 chars): {sample_text[:100]}...")

# Get intermediate values
result = encode_text_query(model, sample_text, return_intermediate=True)

print(f"\n--- Intermediate values ---")
print(f"LLM hidden states shape: {result['text_features'].shape}")
print(f"LLM hidden states (first token, first 5 dims): {result['text_features'][0, 0, :5].tolist()}")
print(f"Projected X_text shape: {result['X_text'].shape}")
print(f"T_feat (feature slots) shape: {result['T_feat'].shape}")
print(f"Confidence scores: {result['confidence'][0].tolist()}")
print(f"Final z_text shape: {result['z_text'].shape}")

# =============================================================================
# Test multiple queries with intermediate values
# =============================================================================
print("\n" + "=" * 70)
print("Testing different queries - checking confidence distribution")
print("=" * 70)

test_queries = [
    "A cylindrical gear with teeth",
    "L-shaped mounting bracket with holes", 
    "Hexagonal bolt with threaded shaft",
    "Flat rectangular plate",
]

test_embeddings = []
for q in test_queries:
    result = encode_text_query(model, q, return_intermediate=True)
    test_embeddings.append(result["z_text"])
    conf = result["confidence"][0]
    print(f"\nQuery: '{q}'")
    print(f"  Confidence: min={conf.min():.3f}, max={conf.max():.3f}, mean={conf.mean():.3f}")
    print(f"  Active slots (conf > 0.3): {(conf > 0.3).sum().item()}")
    print(f"  z_text first 5: {result['z_text'][0, :5].tolist()}")

# Check pairwise similarities
print("\n--- Pairwise similarities ---")
for i in range(len(test_queries)):
    for j in range(i+1, len(test_queries)):
        sim = torch.mm(test_embeddings[i], test_embeddings[j].T).item()
        print(f"  Q{i+1} vs Q{j+1}: {sim:.4f}")

# Gallery stats
print("\n--- Gallery embedding stats ---")
print(f"BRep: shape={brep_embeddings.shape}, norm={brep_embeddings.norm(dim=1).mean():.4f}")
brep_sample_sim = torch.mm(brep_embeddings[:100], brep_embeddings[:100].T)
print(f"BRep self-sim (100 samples): mean={brep_sample_sim.mean():.4f}, std={brep_sample_sim.std():.4f}")

---
## Cell 7: Retrieval Function

In [ ]:
def retrieve(query_emb: torch.Tensor, gallery_emb: torch.Tensor, 
             gallery_uids: list, k: int = 10):
    """
    Retrieve top-k most similar models.
    
    Args:
        query_emb: Query embedding (1, d)
        gallery_emb: Gallery embeddings (N, d)
        gallery_uids: List of UIDs corresponding to gallery
        k: Number of results
        
    Returns:
        List of (uid, score) tuples
    """
    # Compute similarities
    sim = torch.mm(query_emb, gallery_emb.T).squeeze(0)  # (N,)
    
    # Get top-k
    topk_scores, topk_indices = sim.topk(k)
    
    results = []
    for idx, score in zip(topk_indices.tolist(), topk_scores.tolist()):
        results.append((gallery_uids[idx], score))
    
    return results


print("Retrieval function ready!")

---
## Cell 8: STEP Visualization

In [ ]:
def load_step_as_mesh(uid: int):
    """
    Load STEP file and convert to mesh.
    
    Returns:
        vertices: (N, 3) array
        faces: (M, 3) array of vertex indices
    """
    step_path = Path(PATHS["step_dir"]) / f"{uid}.step"
    
    if not step_path.exists():
        print(f"STEP file not found: {step_path}")
        return None, None
    
    # Read STEP
    reader = STEPControl_Reader()
    status = reader.ReadFile(str(step_path))
    
    if status != IFSelect_RetDone:
        print(f"Failed to read STEP: {step_path}")
        return None, None
    
    reader.TransferRoots()
    shape = reader.OneShape()
    
    # Tessellate
    mesh = BRepMesh_IncrementalMesh(shape, 0.1, False, 0.5, True)
    mesh.Perform()
    
    # Extract triangles
    vertices = []
    faces = []
    vertex_offset = 0
    
    explorer = TopExp_Explorer(shape, TopAbs_FACE)
    while explorer.More():
        face = explorer.Current()
        location = TopLoc_Location()
        triangulation = BRep_Tool.Triangulation(face, location)
        
        if triangulation is not None:
            # Get vertices
            for i in range(1, triangulation.NbNodes() + 1):
                node = triangulation.Node(i)
                vertices.append([node.X(), node.Y(), node.Z()])
            
            # Get faces (triangles)
            for i in range(1, triangulation.NbTriangles() + 1):
                tri = triangulation.Triangle(i)
                i1, i2, i3 = tri.Get()
                faces.append([
                    i1 - 1 + vertex_offset,
                    i2 - 1 + vertex_offset,
                    i3 - 1 + vertex_offset
                ])
            
            vertex_offset += triangulation.NbNodes()
        
        explorer.Next()
    
    if not vertices:
        return None, None
    
    return np.array(vertices), np.array(faces)


def visualize_3d(vertices, faces, title="", width=500, height=400):
    """
    Render mesh using Plotly.
    """
    if vertices is None or faces is None:
        return None
    
    fig = go.Figure(data=[
        go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color='lightblue',
            opacity=1.0,
            flatshading=True,
        )
    ])
    
    fig.update_layout(
        title=title,
        width=width,
        height=height,
        scene=dict(
            aspectmode='data',
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor='white',
        ),
        margin=dict(l=0, r=0, t=30, b=0),
    )
    
    return fig


print("STEP visualization ready!")

---
## Cell 9: Results Display

In [ ]:
def get_model_info(uid: int) -> dict:
    """Get metadata for a model."""
    try:
        row = metadata_df.loc[uid]
        return {
            "title": row.get("title", "N/A")[:80],
            "description": str(row.get("description", "N/A"))[:150] + "...",
        }
    except:
        return {"title": "N/A", "description": "N/A"}


def display_results(results: list, show_3d: bool = True, max_3d: int = 3):
    """
    Display retrieval results.
    
    Args:
        results: List of (uid, score) tuples
        show_3d: Whether to render 3D models
        max_3d: Max number of 3D renders
    """
    # Build results table
    table_html = "<table style='width:100%; border-collapse:collapse;'>"
    table_html += "<tr style='background:#f0f0f0;'>"
    table_html += "<th>Rank</th><th>UID</th><th>Score</th><th>Title</th><th>Description</th>"
    table_html += "</tr>"
    
    for rank, (uid, score) in enumerate(results, 1):
        info = get_model_info(uid)
        table_html += f"<tr style='border-bottom:1px solid #ddd;'>"
        table_html += f"<td>{rank}</td>"
        table_html += f"<td>{uid}</td>"
        table_html += f"<td>{score:.4f}</td>"
        table_html += f"<td>{info['title']}</td>"
        table_html += f"<td style='font-size:0.9em;'>{info['description']}</td>"
        table_html += "</tr>"
    
    table_html += "</table>"
    display(HTML(table_html))
    
    # Render 3D models
    if show_3d:
        print(f"\nRendering top {min(max_3d, len(results))} models...")
        for rank, (uid, score) in enumerate(results[:max_3d], 1):
            info = get_model_info(uid)
            vertices, faces = load_step_as_mesh(uid)
            if vertices is not None:
                title = f"#{rank} (UID {uid}, score={score:.3f}): {info['title'][:40]}"
                fig = visualize_3d(vertices, faces, title)
                if fig:
                    fig.show()


print("Results display ready!")

---
## Cell 10: Main Search Function

In [ ]:
def text_to_cad(query: str, modality: str = "brep", k: int = 10, 
                show_3d: bool = True, max_3d: int = 3):
    """
    Search CAD library with natural language.
    
    Args:
        query: Text description (e.g., "cylindrical gear with teeth")
        modality: "brep" or "pc" - which embeddings to search
        k: Number of results to return
        show_3d: Whether to render 3D models
        max_3d: Max 3D models to render
        
    Returns:
        List of (uid, score) results
    """
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"Modality: {modality.upper()}")
    print(f"{'='*70}\n")
    
    # Encode query
    query_emb = encode_text_query(model, query)
    
    # Select gallery
    if modality == "brep":
        gallery = brep_embeddings
    elif modality == "pc":
        gallery = pc_embeddings
    else:
        raise ValueError(f"Unknown modality: {modality}")
    
    # Retrieve
    results = retrieve(query_emb, gallery, gallery_uids, k=k)
    
    # Display
    display_results(results, show_3d=show_3d, max_3d=max_3d)
    
    return results


print("Search function ready!")
print("\nUsage: text_to_cad('your query here')")

---
---
# Interactive Demo
---

In [ ]:
# Example 1: Gear
text_to_cad("A cylindrical gear with helical teeth", modality="brep", k=5)

In [ ]:
# Example 2: Bracket
text_to_cad("L-shaped mounting bracket with holes", modality="brep", k=5)

In [ ]:
# Example 3: Bolt
text_to_cad("Hexagonal bolt with threaded shaft", modality="brep", k=5)

In [ ]:
# Interactive query - enter your search text!
query = input("Enter your CAD search query: ")
if query.strip():
    text_to_cad(query, modality="brep", k=10, show_3d=True)
else:
    print("No query entered. Example usage:")
    print('  text_to_cad("cylindrical shaft with keyway")')

---
## Compare BRep vs PC Retrieval

In [ ]:
def compare_modalities(query: str, k: int = 5):
    """
    Compare retrieval results from BRep and PC embeddings.
    """
    print(f"\n{'='*70}")
    print(f"COMPARING MODALITIES")
    print(f"Query: '{query}'")
    print(f"{'='*70}")
    
    # Encode query
    query_emb = encode_text_query(model, query)
    
    # Retrieve from both
    brep_results = retrieve(query_emb, brep_embeddings, gallery_uids, k=k)
    pc_results = retrieve(query_emb, pc_embeddings, gallery_uids, k=k)
    
    # Build comparison table
    print("\nRank | BRep (UID, Score)      | PC (UID, Score)")
    print("-" * 60)
    
    for i in range(k):
        brep_uid, brep_score = brep_results[i]
        pc_uid, pc_score = pc_results[i]
        
        # Highlight if different
        match = " " if brep_uid == pc_uid else "*"
        
        print(f"  {i+1}  | {brep_uid:>8} ({brep_score:.4f})   | {pc_uid:>8} ({pc_score:.4f}) {match}")
    
    # Count overlap
    brep_set = set(uid for uid, _ in brep_results)
    pc_set = set(uid for uid, _ in pc_results)
    overlap = len(brep_set & pc_set)
    
    print(f"\nOverlap: {overlap}/{k} models appear in both")
    
    return brep_results, pc_results


# Compare modalities
compare_modalities("A flanged bearing housing", k=10)

---
## Compare Different Checkpoints

In [ ]:
def compare_checkpoints(query: str, checkpoint_paths: list, ablation_type: str = "asymmetric_grounding", k: int = 5):
    """
    Compare retrieval results across different checkpoints.
    
    NOTE: This reloads models for each checkpoint (slow).
    For production use, pre-compute embeddings for each checkpoint.
    """
    print(f"\n{'='*70}")
    print(f"COMPARING CHECKPOINTS")
    print(f"Query: '{query}'")
    print(f"{'='*70}")
    
    all_results = {}
    
    for ckpt_path in checkpoint_paths:
        ckpt_name = Path(ckpt_path).stem
        print(f"\nLoading {ckpt_name}...")
        
        # Load model
        ckpt_model, _ = load_model(ckpt_path, ablation_type)
        
        # Encode query
        query_emb = encode_text_query(ckpt_model, query)
        
        # Retrieve (using pre-computed gallery - note: this assumes same model architecture)
        results = retrieve(query_emb, brep_embeddings, gallery_uids, k=k)
        all_results[ckpt_name] = results
        
        # Cleanup
        del ckpt_model
        torch.cuda.empty_cache()
    
    # Print comparison
    print(f"\n{'Rank':<6}", end="")
    for name in all_results:
        print(f"{name[:15]:<18}", end="")
    print()
    print("-" * (6 + 18 * len(all_results)))
    
    for i in range(k):
        print(f"{i+1:<6}", end="")
        for name, results in all_results.items():
            uid, score = results[i]
            print(f"{uid} ({score:.3f})    ", end="")
        print()
    
    return all_results


# Example: Compare epoch 35 vs epoch 70
# compare_checkpoints(
#     "A cylindrical shaft with keyway",
#     [
#         "outputs/ablations/asymmetric_grounding/checkpoint_epoch35.pt",
#         "outputs/ablations/asymmetric_grounding/checkpoint_epoch70.pt",
#     ],
#     k=5
# )